# Module 01 — LLM Basics

**Goal:** Make your first LLM API call and understand every part of it.

This notebook uses **Ollama running locally** with `llama3.2:1b` — no API key, no cloud, no cost.

By the end you will know:
- What `messages` are and why roles matter
- What the raw API response object looks like
- Why we set `temperature=0` for deterministic tasks like SQL generation

---
**Run each cell in order (Shift+Enter).**

## Prerequisites — Install Ollama and pull the model

You only need to do this once.

**1. Install Ollama** — download from https://ollama.com or via terminal:
```bash
# macOS
brew install ollama

# Linux
curl -fsSL https://ollama.com/install.sh | sh
```

**2. Start the Ollama server** (keep this running in a separate terminal):
```bash
ollama serve
```

**3. Pull the model** (~1 GB download, one-time):
```bash
ollama pull llama3.2:1b
```

**4. Verify it works** (should print a short response):
```bash
ollama run llama3.2:1b "Say hello in one word"
```

Once Ollama is running, continue with the cells below.

In [3]:
# Verify Ollama is reachable before we start
import urllib.request

try:
    with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
        print(f"✓ Ollama is running (status {r.status})")
except Exception as e:
    print(f"✗ Ollama not reachable: {e}")
    print("  → Start it with: ollama serve")

✓ Ollama is running (status 200)


## Step 1 — Setup

Ollama exposes an **OpenAI-compatible API** at `http://localhost:11434/v1`.
That means the exact same Python code works with Ollama, OpenAI, Groq, or any other provider —
you just change the `base_url` and `model` name.

| Provider | `base_url` | `api_key` | `model` |
|----------|-----------|----------|--------|
| **Ollama (local)** | `http://localhost:11434/v1` | `"ollama"` | `"llama3.2:1b"` |
| OpenAI | `https://api.openai.com/v1` | your key | `"gpt-4o-mini"` |
| Groq | `https://api.groq.com/openai/v1` | your key | `"llama-3.1-8b-instant"` |
| LM Studio | `http://localhost:1234/v1` | `"lm-studio"` | model filename |

In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Load .env from the repo root (two levels up) if it exists
load_dotenv(Path("../../.env"))

# Defaults to local Ollama — override by setting these in your .env
BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:11434/v1")
API_KEY  = os.getenv("LLM_API_KEY",  "ollama")   # Ollama ignores this; required by the client
MODEL    = os.getenv("LLM_MODEL",    "llama3.2:1b")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

print(f"Model:    {MODEL}")
print(f"Endpoint: {BASE_URL}")

Model:    llama3.2:1b
Endpoint: http://localhost:11434/v1


## Step 2 — Your first API call

The minimum required fields are `model` and `messages`.

`messages` is a list of conversation turns. Each turn has:
- `role`: who is speaking — `"system"`, `"user"`, or `"assistant"`
- `content`: what they said

We'll start with just a `user` message — no system prompt yet.

In [5]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is 2 + 2? Answer in one word."}
    ],
)

# The model's answer is nested at this path:
print(response.choices[0].message.content)

Four.


## Step 3 — Inspect the full response

There's a lot more in the response than just the text.
Let's look at the fields that matter.

In [6]:
print("=== Full response object ===")
print(response)
print()
print("=== Fields you care about ===")
print(f"model used:      {response.model}")
print(f"finish_reason:   {response.choices[0].finish_reason}")
print(f"content:         {response.choices[0].message.content}")
print(f"role:            {response.choices[0].message.role}")
print(f"prompt tokens:   {response.usage.prompt_tokens}")
print(f"response tokens: {response.usage.completion_tokens}")

=== Full response object ===
ChatCompletion(id='chatcmpl-423', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Four.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1776798606, model='llama3.2:1b', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=3, prompt_tokens=38, total_tokens=41, completion_tokens_details=None, prompt_tokens_details=None))

=== Fields you care about ===
model used:      llama3.2:1b
finish_reason:   stop
content:         Four.
role:            assistant
prompt tokens:   38
response tokens: 3


**`finish_reason`** tells you *why* the model stopped generating:
- `"stop"` — normal end of response
- `"length"` — hit the `max_tokens` limit
- `"tool_calls"` — the model wants to call a function (Module 03!)

In Module 04 (the agentic loop), we check `finish_reason` on every turn
to know whether the agent is done or still working.

> **Note on `llama3.2:1b`:** This is a small model (~1 GB). It's great for learning
> because it runs on any laptop, but it's less capable than larger models.
> When you move to production, swap `llama3.2:1b` for `llama3.2:3b`, `llama3.1:8b`,
> or a cloud model — the API code stays identical.

## Step 4 — The system prompt

The `system` role gives the model persistent instructions that apply to every user message.
It's the most powerful knob for controlling model behavior.

In [7]:
def ask(question: str, system: str = "") -> str:
    """Helper: send a question and return the response text."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": question})
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


# Without a system prompt:
print("No system prompt:")
print(ask("Tell me about Paris."))
print()

# With a system prompt that constrains the output:
print("With system prompt (one sentence only):")
print(ask(
    "Tell me about Paris.",
    system="You are a tour guide. Answer every question in exactly one sentence."
))

No system prompt:
Paris, the City of Light. Known for its stunning architecture, romantic atmosphere, and rich cultural heritage, Paris is a must-visit destination for anyone traveling to Europe.

Located in the north-central part of France, Paris is situated along the Seine River and covers an area of approximately 2,039 square kilometers (786 square miles). The city has a population of around 2.1 million people and is served by two international airports: Charles de Gaulle Airport and Orly Airport.

Here are some highlights of what you can experience in Paris:

**Landmarks and Architecture**

Paris is famous for its iconic landmarks, including the Eiffel Tower ( Built for the World's Fair in 1889, it was intended to be a temporary structure but became an instant icon of the city), the Louvre Museum, Notre-Dame Cathedral, Arc de Triomphe, and the Pont des Arts. The city also boasts beautiful gardens, such as the Luxembourg Gardens and Tuileries Garden.

**Food and Wine**

French cuisi

## Step 5 — Temperature: determinism vs creativity

`temperature=0` always picks the most likely next token → same question, same answer every time.  
`temperature=1` samples from the distribution → same question, different answers each run.

For SQL generation we always use `temperature=0`.  
For creative tasks (brainstorming, summarization) you'd use 0.7–1.0.

> **Tip:** Run the cell below multiple times to see the effect.

In [8]:
question = "Name one random country."

print("temperature=0 (deterministic) — should be the same every run:")
for _ in range(3):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0,
    )
    print(" ", r.choices[0].message.content)

print()
print("temperature=1 (creative) — should vary:")
for _ in range(3):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=1,
    )
    print(" ", r.choices[0].message.content)

temperature=0 (deterministic) — should be the same every run:
  Afghanistan.
  Afghanistan.
  Afghanistan.

temperature=1 (creative) — should vary:
  Namibia.
  India.
  Japan.


## Step 6 — Multi-turn conversation

LLMs are **stateless** — they have no memory between API calls.
You simulate memory by appending every turn to `messages` before the next call.

In [9]:
messages = [{"role": "system", "content": "You are a helpful assistant."}]

def chat(user_message: str) -> str:
    """Add a user message, get a response, append both to history."""
    messages.append({"role": "user", "content": user_message})
    response = client.chat.completions.create(model=MODEL, messages=messages)
    assistant_message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_message})
    return assistant_message


print("Turn 1:", chat("My name is Alice."))
print()
print("Turn 2:", chat("What is my name?"))  # should remember 'Alice'
print()
print(f"Messages in history: {len(messages)} (grows with every turn!)")

Turn 1: Hello Alice, how can I help you today?

Turn 2: Alice, that's your birth name. Is everything alright? Do you want to tell me something or just chat for a bit?

Messages in history: 5 (grows with every turn!)


## Step 7 — Swap to a different model (optional)

Try a larger local model or a cloud provider — the code doesn't change at all.

In [ ]:
# Larger local model (pull first: ollama pull llama3.2:3b)
# client_3b = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# MODEL_3B = "llama3.2:3b"

# OpenAI cloud (needs LLM_API_KEY in your .env)
# client_openai = OpenAI(base_url="https://api.openai.com/v1", api_key=os.getenv("LLM_API_KEY"))
# MODEL_OPENAI = "gpt-4o-mini"

# Groq cloud — free tier available at console.groq.com
# client_groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.getenv("GROQ_API_KEY"))
# MODEL_GROQ = "llama-3.1-8b-instant"

# All of the above use the SAME API call:
# response = client_XYZ.chat.completions.create(model=MODEL_XYZ, messages=[...])

print("Uncomment one of the blocks above and re-run to compare outputs across providers.")

## Summary

| Concept | Key point |
|---------|----------|
| `messages` | The full conversation history sent on every call |
| `system` role | Persistent instructions that shape every response |
| `finish_reason` | Why the model stopped — important in the agentic loop (Module 04) |
| `temperature=0` | Deterministic output — use this for SQL generation |
| Stateless | LLMs don't remember — you must append history yourself |
| Provider-agnostic | Same Python code works with Ollama, OpenAI, Groq, LM Studio |

**Next:** Module 02 — how to get reliable *structured* output (JSON) from an LLM instead of free-form text.